# 15. CNN 실습 — 이미지 분류

> **제15장** · **이론편 대응: 12장 (CNN)**
> **예상 소요**: 70분 (데이터 다운로드 1분 + 학습 10분)
> **필요 사양**: **[CPU]** 로 실행 가능 (GPU 있으면 더 빠름)
> **다운로드**: FashionMNIST 약 30MB (자동)

---

## 이 장에서 하는 일

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | **데이터셋 내려받기와 살펴보기** | — |
| 2 | **합성곱 직접 구현 → 이론편 값 (0, −210) 검증** ★ | 12.2절 |
| 3 | PyTorch `Conv2d`와 대조 | 12.2절 |
| 4 | 풀링과 패딩 | 12.3절 |
| 5 | CNN 모델 만들기 | 12.3절 |
| 6 | 학습과 평가 | 12.4절 |
| 7 | **MLP와 비교 — 왜 CNN인가** | 12.1절 |
| 8 | 학습된 필터 들여다보기 | 12.3절 |

1절에서 **데이터를 어디서 어떻게 가져오는지** 자세히 다룬다.
지금까지는 데이터를 코드로 만들어 썼지만, 이제부터는 실제 데이터를 다룬다.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} / 장치: {device}")

try:
    import torchvision
    print(f"torchvision {torchvision.__version__}")
except ImportError:
    print("[필요] torchvision 설치")
    print("  pip install torchvision --index-url https://download.pytorch.org/whl/cu128")

---

## 1. 데이터셋 내려받기 — 어디서 어떻게 오는가

### 이 장에서 쓰는 데이터: FashionMNIST

| 항목 | 내용 |
|---|---|
| 이름 | FashionMNIST |
| 만든 곳 | Zalando Research |
| 내용 | 의류 흑백 사진 28×28 |
| 클래스 | 10개 (티셔츠·바지·가방 등) |
| 학습 데이터 | 60,000장 |
| 시험 데이터 | 10,000장 |
| 용량 | 약 30MB |
| 원본 주소 | `https://github.com/zalandoresearch/fashion-mnist` |

**왜 MNIST(손글씨 숫자)가 아니라 FashionMNIST인가?**
MNIST는 너무 쉬워서 단순한 모델로도 99%가 나온다. 그러면 CNN의 장점이 드러나지 않는다.
FashionMNIST는 같은 크기·형식이면서 훨씬 어려워, 모델 차이를 비교하기에 적합하다.

### 내려받는 방법

`torchvision.datasets`가 다운로드부터 압축 해제까지 자동으로 처리한다.

```python
from torchvision import datasets, transforms

train_data = datasets.FashionMNIST(
    root="./data",        # 저장할 폴더
    train=True,           # 학습용(True) / 시험용(False)
    download=True,        # 없으면 내려받기
    transform=transforms.ToTensor(),   # 이미지 → 텐서 변환
)
```

**각 인자의 뜻**

| 인자 | 설명 |
|---|---|
| `root` | 저장 위치. 폴더가 없으면 만든다 |
| `train` | 학습용/시험용 구분 |
| `download` | `True`면 없을 때만 내려받는다. 이미 있으면 건너뛴다 |
| `transform` | 이미지를 어떻게 변환할지 |

**이미 받은 데이터는 다시 받지 않는다.** `download=True`로 두어도 파일이 있으면 넘어가므로,
여러 번 실행해도 문제없다.

In [ ]:
import os
from pathlib import Path
from torchvision import datasets, transforms

# 저장 위치: 프로젝트 루트의 data 폴더
root = Path.cwd()
if root.name.startswith("part"):
    root = root.parent
data_dir = root / "data"

print("=" * 60)
print("데이터셋 내려받기")
print("=" * 60)
print(f"저장 위치: {data_dir}")
print()

# ToTensor(): PIL 이미지를 텐서로 바꾸고 0~255를 0~1로 나눈다
transform = transforms.ToTensor()

print("내려받는 중... (처음 한 번만, 약 30MB)")
train_data = datasets.FashionMNIST(
    root=str(data_dir), train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(
    root=str(data_dir), train=False, download=True, transform=transform)

print()
print(f"학습 데이터: {len(train_data):,}장")
print(f"시험 데이터: {len(test_data):,}장")
print()
print("클래스 목록")
for i, name in enumerate(train_data.classes):
    print(f"  {i}: {name}")

In [ ]:
from pathlib import Path

root = Path.cwd()
if root.name.startswith("part"):
    root = root.parent
data_dir = root / "data"

print("=" * 60)
print("내려받은 파일 구조")
print("=" * 60)

total = 0
for path in sorted(data_dir.rglob("*")):
    if path.is_file():
        size_mb = path.stat().st_size / 1024**2
        total += size_mb
        rel = path.relative_to(data_dir)
        print(f"  {str(rel):<48}{size_mb:>7.1f} MB")

print("-" * 60)
print(f"  {'합계':<48}{total:>7.1f} MB")
print()
print(".gz 파일은 내려받은 원본이고, 확장자 없는 파일은 압축을 푼 것이다.")
print("한 번 받아 두면 다음부터는 이 폴더에서 바로 읽는다.")
print()
print("주의: data/ 폴더는 .gitignore 에 넣어 저장소에 올리지 않는다.")

### 데이터가 실제로 어떻게 생겼는지 확인

**새 데이터를 받으면 반드시 눈으로 확인한다.** 모양·값 범위·레이블 분포를 보지 않고
바로 학습에 들어가면, 나중에 원인 모를 문제를 겪는다.

In [ ]:
import numpy as np
import torch

print("=" * 55)
print("데이터 하나 살펴보기")
print("=" * 55)

image, label = train_data[0]

print(f"이미지 자료형 : {type(image).__name__}")
print(f"이미지 모양   : {tuple(image.shape)}   ← (채널, 높이, 너비)")
print(f"값의 범위     : {image.min():.3f} ~ {image.max():.3f}")
print(f"레이블        : {label}  ({train_data.classes[label]})")
print()

print("채널이 1인 이유: 흑백이라 색 정보가 하나뿐")
print("  컬러 이미지라면 (3, H, W) — R·G·B 세 채널 (이론편 4.1절)")
print()

# 값의 범위가 0~1인 이유
print("값이 0~1인 이유: ToTensor()가 255로 나눠 주기 때문")
print("  원본 픽셀값은 0~255 정수 (이론편 4.1절)")
print()

# 레이블 분포 확인 — 불균형하지 않은지
labels = train_data.targets.numpy()
counts = np.bincount(labels)
print("클래스별 개수")
for i, (name, cnt) in enumerate(zip(train_data.classes, counts)):
    print(f"  {i} {name:<14}{cnt:>6,}장")
print(f"  {'합계':<17}{counts.sum():>6,}장")
print()
print("각 클래스가 6,000장씩 고르게 있다 — 07장에서 다룬 불균형 문제가 없다.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 5, figsize=(12, 5.5))

# 각 클래스에서 한 장씩 골라 보여준다
labels_np = train_data.targets.numpy()
for cls in range(10):
    idx = np.where(labels_np == cls)[0][0]
    img, lab = train_data[idx]
    ax = axes[cls // 5, cls % 5]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(f"{cls}: {train_data.classes[lab]}", fontsize=9)
    ax.axis("off")

fig.suptitle("FashionMNIST — 클래스별 예시", fontsize=13)
plt.tight_layout()
plt.show()

print("사람이 봐도 헷갈리는 것들이 있다 (셔츠 vs 티셔츠 vs 풀오버).")
print("이래서 MNIST보다 어렵고, 모델 성능 차이가 잘 드러난다.")

---

## 2. 합성곱 직접 구현 — 이론편 12.2절 값 검증 ★

이론편 12.2절에서 세로 경계 검출 필터를 손으로 계산했다. 그 값을 코드로 확인한다.

입력은 왼쪽이 어둡고(10) 오른쪽이 밝은(80) 5×5 이미지,
필터는 왼쪽 열 +1 / 오른쪽 열 −1인 3×3이었다.

**이론편에서 구한 출력**

$$Y = \begin{pmatrix} 0 & -210 & -210 \\ 0 & -210 & -210 \\ 0 & -210 & -210 \end{pmatrix}$$

In [ ]:
import numpy as np


def conv2d_manual(image, kernel):
    """합성곱을 반복문으로 직접 구현 (이론편 12.2절)

    필터를 한 칸씩 옮기며, 겹치는 부분끼리 곱해 더한다.
    """
    ih, iw = image.shape
    kh, kw = kernel.shape
    oh, ow = ih - kh + 1, iw - kw + 1      # 패딩 없을 때의 출력 크기

    out = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            patch = image[i:i+kh, j:j+kw]   # 겹치는 영역
            out[i, j] = (patch * kernel).sum()
    return out


# 이론편 12.2절과 완전히 같은 입력
X_img = np.array([[10, 10, 10, 80, 80]] * 5, dtype=float)
K = np.array([[1, 0, -1]] * 3, dtype=float)

print("=" * 60)
print("이론편 12.2절 값 검증")
print("=" * 60)
print("입력 X (왼쪽 어두움 10 / 오른쪽 밝음 80)")
print(X_img.astype(int))
print()
print("필터 K (세로 경계 검출)")
print(K.astype(int))
print()

Y = conv2d_manual(X_img, K)
print("출력 Y")
print(Y.astype(int))
print()
print("이론편 값")
print("[[   0 -210 -210]")
print(" [   0 -210 -210]")
print(" [   0 -210 -210]]")
print("-" * 60)

expected = np.array([[0, -210, -210]] * 3, dtype=float)
assert np.allclose(Y, expected), "이론편 값과 다릅니다"
print("[OK] 이론편 12.2절 손계산과 일치")

In [ ]:
import numpy as np

print("=" * 60)
print("한 칸씩 계산 과정 따라가기")
print("=" * 60)

for j in range(3):
    patch = X_img[0:3, j:j+3]
    left = patch[:, 0].sum()      # 필터 +1이 곱해지는 열
    right = patch[:, 2].sum()     # 필터 -1이 곱해지는 열
    value = left - right

    print(f"\n[0, {j}] 위치")
    print(f"  영역:\n{patch.astype(int)}")
    print(f"  왼쪽 열 합 = {left:.0f}  (필터 +1)")
    print(f"  오른쪽 열 합 = {right:.0f}  (필터 -1)")
    print(f"  결과 = {left:.0f} - {right:.0f} = {value:.0f}")
    if value == 0:
        print("  → 균일한 영역. 경계 없음")
    else:
        print("  → 경계 있음. 음수이므로 '왼쪽이 어둡다'")

print()
print("=" * 60)
print("이 필터가 하는 일: 좌우 밝기 차이를 재는 것")
print("사람이 알려주지 않았는데도 '경계 지도'가 만들어졌다.")

### 파라미터 수 비교 — 이론편 12.2절

이론편에서 완전연결과 합성곱의 파라미터 수를 비교했다. 다시 확인한다.

In [ ]:
print("=" * 55)
print("파라미터 수 (이론편 12.2절)")
print("=" * 55)

in_size = 5 * 5      # 입력 25개
out_size = 3 * 3     # 출력 9개
fc_params = in_size * out_size
conv_params = 3 * 3

print(f"{'방식':<24}{'파라미터':<14}{'설명'}")
print("-" * 55)
print(f"{'완전연결 (25 → 9)':<24}{fc_params:<14}{'모든 입력-출력 쌍마다'}")
print(f"{'합성곱 (3x3 필터)':<24}{conv_params:<14}{'같은 필터를 9곳에 재사용'}")
print("-" * 55)
print(f"비율: {fc_params / conv_params:.0f}배 차이")
print()
print("실제 이미지 크기(28x28)로 계산하면 차이가 훨씬 커진다:")
fc_real = (28*28) * (26*26)
print(f"  완전연결: {fc_real:,}개")
print(f"  합성곱  : {conv_params}개  ({fc_real/conv_params:,.0f}배)")
print()
print("이것이 이론편 12.1절에서 다룬 '완전연결의 한계'다.")

---

## 3. PyTorch `Conv2d`와 대조

직접 만든 것과 PyTorch가 같은 계산을 하는지 확인한다.

PyTorch의 합성곱은 **4차원 텐서**를 받는다.

$$(\text{배치}, \text{채널}, \text{높이}, \text{너비})$$

우리 예제는 이미지 1장, 채널 1개이므로 `(1, 1, 5, 5)` 모양으로 바꿔야 한다.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

print("=" * 60)
print("직접 구현 vs PyTorch conv2d")
print("=" * 60)

# NumPy → PyTorch 텐서, 4차원으로 변형
x_t = torch.tensor(X_img, dtype=torch.float32).view(1, 1, 5, 5)
k_t = torch.tensor(K, dtype=torch.float32).view(1, 1, 3, 3)

print(f"입력 모양 : {tuple(x_t.shape)}   ← (배치, 채널, 높이, 너비)")
print(f"필터 모양 : {tuple(k_t.shape)}   ← (출력채널, 입력채널, 높이, 너비)")
print()

out_torch = F.conv2d(x_t, k_t)
print(f"출력 모양 : {tuple(out_torch.shape)}")
print()
print("PyTorch 결과")
print(out_torch.view(3, 3).numpy().astype(int))
print()
print("직접 구현 결과")
print(Y.astype(int))
print("-" * 60)

assert np.allclose(Y, out_torch.view(3, 3).numpy())
print("[OK] 두 결과가 일치")
print()
print("주의: 수학에서 말하는 '합성곱(convolution)'은 필터를 뒤집지만,")
print("      딥러닝 라이브러리는 뒤집지 않는 '교차상관'을 쓴다.")
print("      필터를 학습으로 찾으므로 뒤집든 안 뒤집든 결과가 같기 때문이다.")

---

## 4. 패딩과 스트라이드, 풀링 — 이론편 12.3절

합성곱을 적용하면 출력이 입력보다 작아진다(5×5 → 3×3). 층을 여러 개 쌓으면
크기가 계속 줄어드는 문제가 생긴다.

| 개념 | 하는 일 | 효과 |
|---|---|---|
| 패딩(padding) | 가장자리에 0을 두름 | 크기 유지 |
| 스트라이드(stride) | 필터를 몇 칸씩 옮길지 | 크기 축소 |
| 풀링(pooling) | 영역의 최댓값/평균만 취함 | 크기 축소, 위치 변화에 둔감 |

출력 크기는 다음 식으로 계산된다.

$$O = \left\lfloor \frac{I + 2P - K}{S} \right\rfloor + 1$$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print("=" * 65)
print("패딩·스트라이드에 따른 출력 크기")
print("=" * 65)

x = torch.randn(1, 1, 28, 28)
print(f"입력: {tuple(x.shape[2:])}")
print()
print(f"{'설정':<32}{'출력 크기':<16}{'공식 계산'}")
print("-" * 65)

configs = [
    ("커널3, 패딩0, 스트라이드1", 3, 0, 1),
    ("커널3, 패딩1, 스트라이드1", 3, 1, 1),
    ("커널3, 패딩1, 스트라이드2", 3, 1, 2),
    ("커널5, 패딩2, 스트라이드1", 5, 2, 1),
]
for name, k, p, s in configs:
    # ── nn.Conv2d 파라미터 ───────────────────────────────────────
    #   in_channels   입력 채널 수.  흑백=1, 컬러=3, 중간층=이전 출력
    #   out_channels  필터 개수 = 출력 채널.  예: 16, 32, 64, 128
    #   kernel_size   필터 크기.  예: 3(3x3), 5(5x5), (3,5) 도 가능
    #                 3x3 이 가장 흔하다 (작은 필터를 겹쳐 쓰는 방식)
    #   stride        이동 간격.  기본값 1
    #                 2 로 하면 출력 크기가 절반 (풀링 대신 쓰기도)
    #   padding       가장자리 채움.  기본값 0
    #                 'same' 또는 kernel_size//2 로 하면 크기 유지
    #   dilation      팽창 계수.  기본값 1.  넓은 시야가 필요할 때
    #   groups        그룹 합성곱.  기본값 1
    #                 in_channels 로 하면 depthwise (경량 모델)
    #   bias          기본값 True.  BatchNorm 뒤따르면 False
    #
    #   출력 크기 = (입력 + 2*padding - kernel) / stride + 1
    # ──────────────────────────────────────────────────────────────
    conv = nn.Conv2d(1, 1, kernel_size=k, padding=p, stride=s)
    out = conv(x)
    formula = (28 + 2*p - k) // s + 1
    print(f"{name:<32}{str(tuple(out.shape[2:])):<16}(28+2*{p}-{k})//{s}+1 = {formula}")

print("-" * 65)
print()
print("커널 3에 패딩 1을 주면 크기가 유지된다 — 가장 흔히 쓰는 설정이다.")
print()

# 풀링
print("=" * 65)
print("풀링")
print("=" * 65)
sample = torch.tensor([[[[1., 3., 2., 4.],
                         [5., 6., 1., 2.],
                         [7., 2., 8., 3.],
                         [1., 4., 2., 9.]]]])
print("입력 4x4")
print(sample.view(4, 4).numpy())
print()
print("MaxPool2d(2) — 2x2 영역의 최댓값")
print(F.max_pool2d(sample, 2).view(2, 2).numpy())
print()
print("AvgPool2d(2) — 2x2 영역의 평균")
print(F.avg_pool2d(sample, 2).view(2, 2).numpy())
print()
print("풀링은 학습할 파라미터가 없다. 단순히 크기를 줄이는 연산이다.")
print("최댓값을 쓰면 '그 영역에 특징이 있었는가'만 남고 정확한 위치는 흐려진다.")
print("→ 이론편 12.3절에서 다룬 위치 변화에 대한 둔감함")

---

## 5. CNN 모델 만들기 — 이론편 12.3절

이제 실제 모델을 만든다. 이론편 12.3절에서 다룬 전형적인 구조를 따른다.

```
입력 (1, 28, 28)
  ↓ Conv 3x3, 16채널 + ReLU
  ↓ MaxPool 2x2          → (16, 14, 14)
  ↓ Conv 3x3, 32채널 + ReLU
  ↓ MaxPool 2x2          → (32, 7, 7)
  ↓ Flatten              → 1568
  ↓ Linear 64 + ReLU
  ↓ Linear 10            → 클래스 10개
```

**채널이 늘고 크기가 주는 패턴**에 주목하자. 이것이 이론편 12.3절에서 다룬
"저수준 특징에서 고수준 특징으로" 가는 구조다.

In [ ]:
import torch
import torch.nn as nn


class SimpleCNN(nn.Module):
    """FashionMNIST 분류용 CNN (이론편 12.3절 구조)"""

    def __init__(self, n_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            # 1블록: 28x28 → 14x14
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # 크기 유지
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 절반으로

            # 2블록: 14x14 → 7x7
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model = SimpleCNN()

print("=" * 60)
print("모델 구조")
print("=" * 60)
print(model)
print()

# 각 단계에서 모양이 어떻게 변하는지 추적
x = torch.randn(1, 1, 28, 28)
print("=" * 60)
print("텐서 모양의 변화")
print("=" * 60)
print(f"{'단계':<28}{'출력 모양'}")
print("-" * 60)
print(f"{'입력':<28}{tuple(x.shape)}")
h = x
for name, layer in model.features.named_children():
    h = layer(h)
    print(f"{type(layer).__name__:<28}{tuple(h.shape)}")
for name, layer in model.classifier.named_children():
    h = layer(h)
    print(f"{type(layer).__name__:<28}{tuple(h.shape)}")
print("-" * 60)

total = sum(p.numel() for p in model.parameters())
print(f"전체 파라미터: {total:,}개")

### `32 * 7 * 7`은 어떻게 나왔나

`nn.Linear(32 * 7 * 7, 64)`에서 입력 크기를 직접 계산해야 한다. 자주 실수하는 부분이다.

| 단계 | 크기 | 이유 |
|---|---|---|
| 입력 | 1 × 28 × 28 | |
| Conv(패딩1) | 16 × 28 × 28 | 패딩 덕분에 크기 유지 |
| MaxPool(2) | 16 × 14 × 14 | 절반 |
| Conv(패딩1) | 32 × 14 × 14 | 크기 유지 |
| MaxPool(2) | 32 × 7 × 7 | 절반 |
| Flatten | **1568** | 32 × 7 × 7 = 1568 |

계산이 헷갈리면 위 셀처럼 **모양을 출력해 확인**하는 것이 확실하다.
크기가 안 맞으면 `RuntimeError: mat1 and mat2 shapes cannot be multiplied` 오류가 난다.

---

## 6. 학습과 평가 — 이론편 12.4절

12장에서 익힌 표준 학습 루프를 그대로 쓴다.

**CPU에서도 돌아가도록** 데이터의 일부만 사용한다. 전체를 쓰면 CPU로 수십 분이 걸린다.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import time

# CPU에서도 빠르게 끝나도록 일부만 사용
N_TRAIN = 6000      # 전체 60,000장 중
N_TEST = 1000       # 전체 10,000장 중

train_subset = Subset(train_data, range(N_TRAIN))
test_subset = Subset(test_data, range(N_TEST))

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=256)

print("=" * 55)
print("학습 설정")
print("=" * 55)
print(f"학습 데이터 : {N_TRAIN:,}장  (전체 {len(train_data):,}장 중)")
print(f"시험 데이터 : {N_TEST:,}장")
print(f"배치 크기   : 64")
print(f"1에폭 스텝  : {len(train_loader)}회")
print(f"장치        : {device}")
print()
print("전체 데이터를 쓰려면 N_TRAIN을 60000으로 바꾸면 된다.")
print("GPU가 있으면 1~2분, CPU면 10분 내외가 걸린다.")

In [ ]:
import torch
import torch.nn as nn
import time

torch.manual_seed(42)      # 재현성 (이론편 11.6절)

model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()          # 다중 분류 (이론편 6.5절)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 5
history = {"train_loss": [], "test_loss": [], "test_acc": []}

print("=" * 60)
print("학습 시작")
print("=" * 60)
t_start = time.time()

for epoch in range(EPOCHS):
    # --- 훈련 ---
    model.train()
    running = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        running += loss.item() * len(xb)
    train_loss = running / len(train_subset)

    # --- 평가 ---
    model.eval()
    correct, test_running = 0, 0.0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            test_running += criterion(out, yb).item() * len(xb)
            correct += (out.argmax(1) == yb).sum().item()

    test_loss = test_running / len(test_subset)
    test_acc = correct / len(test_subset)

    history["train_loss"].append(train_loss)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(f"에폭 {epoch+1}/{EPOCHS} | 훈련손실 {train_loss:.4f} | "
          f"검증손실 {test_loss:.4f} | 정확도 {test_acc:.4f} | "
          f"{time.time()-t_start:.0f}초")

print("-" * 60)
print(f"최종 정확도: {history['test_acc'][-1]:.4f}")
print(f"총 소요 시간: {time.time()-t_start:.0f}초")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

epochs = np.arange(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 03장에서 배운 손실 곡선
ax = axes[0]
ax.plot(epochs, history["train_loss"], marker="o", label="훈련 손실", linewidth=2)
ax.plot(epochs, history["test_loss"], marker="s", label="검증 손실",
        linewidth=2, linestyle="--")
ax.set_xlabel("에폭")
ax.set_ylabel("손실")
ax.set_title("학습 곡선")
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(epochs, history["test_acc"], marker="o", color="#0D9488", linewidth=2)
ax.set_xlabel("에폭")
ax.set_ylabel("정확도")
ax.set_title("검증 정확도")
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

gap = history["train_loss"][-1] - history["test_loss"][-1]
print(f"훈련·검증 손실 차이: {abs(gap):.4f}")
if abs(gap) < 0.1:
    print("→ 격차가 작다. 정상적인 학습이다 (03장 3절).")
else:
    print("→ 격차가 있다. 데이터를 늘리거나 정규화를 고려한다.")

### 혼동행렬로 어디서 틀리는지 보기

07장에서 배운 혼동행렬을 써서 **어떤 클래스를 헷갈리는지** 확인한다.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

model.eval()
all_pred, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        all_pred.append(model(xb).argmax(1).cpu().numpy())
        all_true.append(yb.numpy())

y_pred = np.concatenate(all_pred)
y_true = np.concatenate(all_true)
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
plt.colorbar(im, ax=ax)

names = train_data.classes
ax.set_xticks(range(10)); ax.set_xticklabels(names, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(10)); ax.set_yticklabels(names, fontsize=8)
ax.set_xlabel("예측")
ax.set_ylabel("실제")
ax.set_title("혼동행렬")

for i in range(10):
    for j in range(10):
        if cm[i, j] > 0:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7,
                    color="white" if cm[i, j] > cm.max()/2 else "black")

plt.tight_layout()
plt.show()

# 가장 많이 헷갈린 쌍 찾기
np.fill_diagonal(cm, 0)
idx = np.unravel_index(cm.argmax(), cm.shape)
print(f"가장 많이 헷갈린 경우: 실제 '{names[idx[0]]}' → 예측 '{names[idx[1]]}' ({cm[idx]}건)")
print()
print("사람이 봐도 비슷한 것들끼리 헷갈린다는 점이 흥미롭다.")

---

## 7. MLP와 비교 — 왜 CNN인가 (이론편 12.1절)

이론편 12.1절에서 "완전연결 신경망은 이미지에 부적합하다"고 했다.
**같은 데이터·같은 파라미터 수**로 비교해 확인한다.

In [ ]:
import torch
import torch.nn as nn
import time

def train_and_eval(model, epochs=5, seed=42):
    torch.manual_seed(seed)
    model = model.to(device)
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    t0 = time.time()
    for _ in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()

    model.eval()
    correct = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            correct += (model(xb).argmax(1) == yb).sum().item()
    return correct / len(test_subset), time.time() - t0


# 파라미터 수를 비슷하게 맞춘 MLP
mlp = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28*28, 128), nn.ReLU(),
    nn.Linear(128, 10),
)

print("=" * 65)
print("CNN vs MLP (같은 데이터, 5에폭)")
print("=" * 65)
print(f"{'모델':<12}{'파라미터':<14}{'정확도':<14}{'학습 시간'}")
print("-" * 65)

for name, m in [("MLP", mlp), ("CNN", SimpleCNN())]:
    params = sum(p.numel() for p in m.parameters())
    acc, sec = train_and_eval(m)
    print(f"{name:<12}{params:<14,}{acc:<14.4f}{sec:.0f}초")

print("-" * 65)
print()
print("파라미터 수는 비슷한데 CNN이 더 정확하다.")
print("이유는 이론편 12.1~12.2절에서 다룬 두 가지 성질이다:")
print("  1) 가중치 공유 — 같은 필터를 모든 위치에 재사용")
print("  2) 국소 연결 — 가까운 픽셀끼리만 먼저 본다")

### 위치 변화에 대한 강인함 확인

이론편 12.1절에서 "완전연결은 물체가 조금만 움직여도 다른 입력으로 본다"고 했다.
이미지를 몇 픽셀 옮겨서 두 모델의 정확도가 어떻게 변하는지 본다.

In [ ]:
import torch
import numpy as np

def shift_images(x, dx=3):
    """이미지를 오른쪽으로 dx 픽셀 이동"""
    return torch.roll(x, shifts=dx, dims=3)

print("=" * 60)
print("이미지를 이동시켰을 때 정확도 변화 (이론편 12.1절)")
print("=" * 60)

torch.manual_seed(42)
mlp_trained = nn.Sequential(nn.Flatten(), nn.Linear(28*28, 128),
                            nn.ReLU(), nn.Linear(128, 10))
_ = train_and_eval(mlp_trained)      # 학습

print(f"{'이동량':<12}{'MLP 정확도':<18}{'CNN 정확도'}")
print("-" * 60)

for dx in [0, 2, 4]:
    accs = {}
    for name, m in [("MLP", mlp_trained), ("CNN", model)]:
        m.eval()
        correct = 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = shift_images(xb.to(device), dx)
                correct += (m(xb).argmax(1) == yb.to(device)).sum().item()
        accs[name] = correct / len(test_subset)
    print(f"{dx}픽셀{'':<8}{accs['MLP']:<18.4f}{accs['CNN']:.4f}")

print("-" * 60)
print()
print("둘 다 떨어지지만, 이동에 대한 민감도가 다르다.")
print("CNN의 풀링이 위치 변화를 어느 정도 흡수하기 때문이다 (이론편 12.3절).")
print()
print("※ 학습 시 이동된 이미지를 함께 보여주면(데이터 증강) 더 강해진다.")

---

## 8. 학습된 필터 들여다보기

이론편 12.3절에서 "CNN의 첫 층은 가장자리·색 변화 같은 저수준 특징을 학습한다"고 했다.
실제로 그런지 첫 번째 합성곱 층의 필터를 그려 본다.

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# 첫 번째 Conv2d 층의 가중치
first_conv = model.features[0]
weights = first_conv.weight.detach().cpu().numpy()   # (16, 1, 3, 3)

print("=" * 55)
print("첫 번째 합성곱 층의 필터")
print("=" * 55)
print(f"필터 모양: {weights.shape}   ← (필터 수, 입력채널, 높이, 너비)")
print(f"필터 개수: {weights.shape[0]}개")
print()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    if i < weights.shape[0]:
        w = weights[i, 0]
        ax.imshow(w, cmap="RdBu_r", vmin=-w.std()*2.5, vmax=w.std()*2.5)
        ax.set_title(f"필터 {i}", fontsize=8)
    ax.axis("off")

fig.suptitle("학습된 3x3 필터 (붉은색=양수, 푸른색=음수)", fontsize=12)
plt.tight_layout()
plt.show()

print("2절에서 우리가 손으로 만든 세로 경계 필터와 비슷한 것들이 보인다.")
print("한쪽이 붉고 반대쪽이 푸른 필터가 그런 역할을 한다.")
print()
print("중요한 것은 이 값들을 사람이 정하지 않았다는 사실이다.")
print("역전파로 스스로 찾아낸 것이다 (이론편 10.4절, 12.3절).")

In [ ]:
import torch
import matplotlib.pyplot as plt

# 실제 이미지에 필터를 적용하면 무엇이 나오는지
sample_img, sample_label = test_data[0]
x = sample_img.unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    feat = model.features[0](x)      # 첫 Conv 층만 통과

feat_np = feat.squeeze(0).cpu().numpy()

fig, axes = plt.subplots(2, 9, figsize=(15, 4))

# 원본
axes[0, 0].imshow(sample_img.squeeze(), cmap="gray")
axes[0, 0].set_title(f"원본\n{test_data.classes[sample_label]}", fontsize=8)
axes[0, 0].axis("off")
axes[1, 0].axis("off")

# 각 필터의 출력
for i in range(16):
    r, c = i // 8, i % 8 + 1
    if c < 9:
        axes[r, c].imshow(feat_np[i], cmap="viridis")
        axes[r, c].set_title(f"필터 {i}", fontsize=7)
        axes[r, c].axis("off")

fig.suptitle("첫 번째 합성곱 층의 출력 (특징 지도)", fontsize=12)
plt.tight_layout()
plt.show()

print("각 필터가 원본에서 서로 다른 것을 잡아냈다.")
print("가로 경계를 강조한 것, 세로를 강조한 것, 전체 밝기를 본 것 등이 섞여 있다.")
print()
print("2절에서 우리가 만든 필터가 '경계 지도'를 만든 것과 같은 일이,")
print("16가지 방식으로 동시에 일어나고 있는 것이다.")

---

## 9. 정리

### 확인한 이론편 값

| 이론편 절 | 내용 | 결과 |
|---|---|---|
| 12.2 | 합성곱 출력 (0, −210) | 직접 구현 ✓ |
| 12.2 | PyTorch conv2d와 동일 | 일치 ✓ |
| 12.2 | 파라미터 225개 vs 9개 | 확인 ✓ |
| 12.1 | CNN이 MLP보다 이미지에 유리 | 실험 확인 ✓ |
| 12.3 | 첫 층이 경계를 학습 | 시각화 확인 ✓ |

### 데이터셋 다루기 요약

```python
from torchvision import datasets, transforms

data = datasets.FashionMNIST(
    root="./data",                    # 저장 위치
    train=True,                       # 학습용/시험용
    download=True,                    # 없으면 받기 (있으면 건너뜀)
    transform=transforms.ToTensor(),  # 이미지 → 텐서, 0~1로 정규화
)
```

**새 데이터를 받으면 반드시 확인할 것**

| 항목 | 확인 방법 |
|---|---|
| 모양 | `data[0][0].shape` |
| 값 범위 | `.min()`, `.max()` |
| 클래스 분포 | `np.bincount(targets)` |
| 눈으로 | `plt.imshow()` 로 몇 장 |

### 기억할 것

| 항목 | 요점 |
|---|---|
| 입력 모양 | `(배치, 채널, 높이, 너비)` 4차원 |
| 패딩 1 + 커널 3 | 크기 유지 — 가장 흔한 설정 |
| 풀링 | 파라미터 없음, 크기 축소 + 위치 둔감 |
| `Flatten` 이후 크기 | 직접 계산 필요 — 헷갈리면 출력해 확인 |
| CNN이 유리한 이유 | 가중치 공유 + 국소 연결 |

### 다음 장

**16. RNN/LSTM 실습 — 순서가 있는 데이터** — 이미지가 아닌 **순서가 있는 데이터**를 다룬다.
이론편 13.4절에서 손으로 계산한 **LSTM 셀 상태 0.924**를 확인한다.